In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
    
import torch
from torch import nn
from torch.utils.data import DataLoader
from src.model import ProteinTransformerModel
from src.dataset import (
    ProteinSequenceDataset,
    build_amino_acid_vocab,
    build_label_mapping,
    encode_labels,
    collate_protein_batch,
    load_protein_csv
)

In [2]:
sequences, labels = load_protein_csv("../data/proteins.csv")

vocab = build_amino_acid_vocab()

label_to_index, index_to_label = build_label_mapping(labels)

print("Vocabulary:", vocab)
print("Label to Index Mapping:", label_to_index)
print("Index to Label Mapping:", index_to_label)

Vocabulary: {'<PAD>': 0, '<UNK>': 1, 'A': 2, 'C': 3, 'D': 4, 'E': 5, 'F': 6, 'G': 7, 'H': 8, 'I': 9, 'K': 10, 'L': 11, 'M': 12, 'N': 13, 'P': 14, 'Q': 15, 'R': 16, 'S': 17, 'T': 18, 'V': 19, 'W': 20, 'Y': 21}
Label to Index Mapping: {'dna_binding': 0, 'enzyme': 1, 'membrane': 2, 'structural': 3}
Index to Label Mapping: {0: 'dna_binding', 1: 'enzyme', 2: 'membrane', 3: 'structural'}


In [3]:
encoded_labels = encode_labels(labels, label_to_index)

dataset = ProteinSequenceDataset(
    sequences=sequences,
    labels=encoded_labels,
    vocab=vocab,
)

encoded_sequences, encoded_labels = dataset[0]

print("Encoded Sequence:", encoded_sequences)
print("Encoded Label:", encoded_labels)
print(f"label name: {index_to_label[encoded_labels.item()]}")

Encoded Sequence: tensor([12,  5, 18,  6,  8,  3, 16, 14, 18,  5,  2, 11,  3, 21,  4, 12, 16,  3,
         5, 18, 11,  2, 20, 11,  8, 12,  8,  8, 11, 10, 19, 12, 16,  5, 10,  6,
         7, 10,  5,  2,  8,  7,  7, 11,  8, 19, 13, 19, 12, 10,  6, 11, 20,  9,
         7])
Encoded Label: tensor(1)
label name: enzyme


In [4]:
data_loader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=collate_protein_batch
)
        


In [5]:
sequences_batch, labels_batch, lengths_batch = next(
    iter(data_loader)
)

print("Sequences:", sequences_batch.shape)
print("Labels:", labels_batch.shape)
print("Lengths:", lengths_batch.shape)


Sequences: torch.Size([8, 69])
Labels: torch.Size([8])
Lengths: torch.Size([8])


In [6]:
max_seq_length = 500

model = ProteinTransformerModel(
    vocab_size=len(vocab),
    embedding_dim=128,
    num_classes=len(label_to_index),
    num_heads=4,
    num_layers=2,
    max_seq_length=max_seq_length,
    feedforward_dim=256,
    dropout=0.1
)

logits = model(sequences_batch)

In [7]:
print("Logits:", logits.shape)
print("Labels Batch:", labels_batch.shape)

Logits: torch.Size([8, 4])
Labels Batch: torch.Size([8])


In [8]:
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, labels_batch)

print("Loss:", loss.item())

Loss: 1.5312154293060303


In [9]:
print("input shape:", sequences_batch.shape)
print("labels shape:", labels_batch.shape)
print("lengths shape:", lengths_batch.shape)
print("Output shape:", logits.shape)

input shape: torch.Size([8, 69])
labels shape: torch.Size([8])
lengths shape: torch.Size([8])
Output shape: torch.Size([8, 4])


In [10]:
loss.backward()

print(model.classifier is not None)
print("Classifier weights shape:", model.classifier.weight.shape)

True
Classifier weights shape: torch.Size([4, 128])
